In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Data: true labels and predicted scores
y_true = np.array([1, 1, 1, 0, 1, 0, 0, 1, 0, 0])
y_scores = np.array([0.95, 0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25, 0.15, 0.05])

# Initialize lists to store metrics
precisions = []
recalls = []
tprs = []
fprs = []
thresholds = np.append(np.unique(y_scores), 1.0)  # Include 1.0

In [3]:
thresholds

array([0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95, 1.  ])

In [4]:

# Compute metrics for each threshold
for thresh in thresholds:
    y_pred = (y_scores >= thresh).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tpr = recall
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    precisions.append(precision)
    recalls.append(recall)
    tprs.append(tpr)
    fprs.append(fpr)

In [7]:
# Create subplots for PR and ROC curves
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Precision-Recall Curve", "ROC Curve"),
                    specs=[[{"type": "xy"}, {"type": "xy"}]])

# PR Curve
fig.add_trace(
    go.Scatter(
        x=recalls, y=precisions, mode='lines+markers',
        name='PR Curve',
        hovertemplate='Threshold: %{customdata:.2f}<br>Recall: %{x:.3f}<br>Precision: %{y:.3f}',
        customdata=thresholds
    ),
    row=1, col=1
)

# Annotate selected thresholds on PR Curve
selected_thresholds = [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95, 1.]
for thresh in selected_thresholds:
    idx = np.where(thresholds == thresh)[0][0]
    fig.add_annotation(
        x=recalls[idx], y=precisions[idx],
        text=f'T={thresh:.2f}',
        showarrow=True, arrowhead=1, ax=20, ay=-20,
        row=1, col=1
    )

# ROC Curve
fig.add_trace(
    go.Scatter(
        x=fprs, y=tprs, mode='lines+markers',
        name='ROC Curve',
        hovertemplate='Threshold: %{customdata:.2f}<br>FPR: %{x:.3f}<br>TPR: %{y:.3f}',
        customdata=thresholds
    ),
    row=1, col=2
)

# Random classifier line for ROC
fig.add_trace(
    go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines',
        name='Random Classifier', line=dict(dash='dash', color='gray')
    ),
    row=1, col=2
)

# Annotate selected thresholds on ROC Curve
for thresh in selected_thresholds:
    idx = np.where(thresholds == thresh)[0][0]
    fig.add_annotation(
        x=fprs[idx], y=tprs[idx],
        text=f'T={thresh:.2f}',
        showarrow=True, arrowhead=1, ax=20, ay=-20,
        row=1, col=2
    )

# Update layout
fig.update_layout(
    title_text="PR and ROC Curves with Threshold Annotations",
    showlegend=True,
    width=1200, height=600
)

# Update axes
fig.update_xaxes(title_text="Recall", row=1, col=1)
fig.update_yaxes(title_text="Precision", row=1, col=1)
fig.update_xaxes(title_text="False Positive Rate", row=1, col=2)
fig.update_yaxes(title_text="True Positive Rate", row=1, col=2)

# Show plot
fig.show()

# Print table for reference
print("Threshold | Precision | Recall | TPR | FPR")
for t, p, r, tpr, fpr in zip(thresholds, precisions, recalls, tprs, fprs):
    print(f"{t:.2f}     | {p:.3f}    | {r:.3f}  | {tpr:.3f} | {fpr:.3f}")

Threshold | Precision | Recall | TPR | FPR
0.05     | 0.500    | 1.000  | 1.000 | 1.000
0.15     | 0.556    | 1.000  | 1.000 | 0.800
0.25     | 0.625    | 1.000  | 1.000 | 0.600
0.35     | 0.571    | 0.800  | 0.800 | 0.600
0.45     | 0.667    | 0.800  | 0.800 | 0.400
0.55     | 0.800    | 0.800  | 0.800 | 0.200
0.65     | 0.750    | 0.600  | 0.600 | 0.200
0.75     | 1.000    | 0.600  | 0.600 | 0.000
0.85     | 1.000    | 0.400  | 0.400 | 0.000
0.95     | 1.000    | 0.200  | 0.200 | 0.000
1.00     | 1.000    | 0.000  | 0.000 | 0.000
